# Iniciando o Spark

In [1]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_atraso") \
    .getOrCreate()

# Carregando pacotes

In [3]:
# Pacotes de manipulacao
import sys
import os
from pyspark.sql.functions import col, least
#import pandas as pd
#import numpy as np

# Pacotes de visualizacao
#import matplotlib.pyplot as plt
#import seaborn as sns

#print("pandas:", pd.__version__)
#print("numpy:", np.__version__)

# Conexão ao repositório via gdrive

In [4]:

# Acesso aos módulos do diretório Colab x Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

pasta_in = 'base_score_bureau_movel_full/'
pasta_out = 'Feature_store/'
path_padrao = "/content/gdrive/Othercomputers/Meu laptop"

bucket_trusted = f"{path_padrao}/Database_trusted/{pasta_in}"
bucket_feature_store =f"{path_padrao}/{pasta_out}"

Mounted at /content/gdrive


# Conexão ao repositório via OCI

In [ ]:
# Buckets e nomes de saída nuvem = "oci://"
#pasta_in = 'base_score_bureau_movel_full/'
#pasta_out = 'Feature_store/'
#namespace = "@grxzqsiaote6/"
#bucket_trusted = f"oci://TRUSTED{namespace}{pasta_in}"
#bucket_feature_store = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_out}"

# Carregando databases

#### Base Dados Cadastrais

In [5]:
## Carregando todos arquivos em parquet de uma pasta
#path = project_root +'database/raw/base_score_bureau_movel/base_score_bureau_movel/'

#all_files = [os.path.join('database/raw/base_score_bureau_movel/', f) for f in os.listdir(project_root/'database/raw/base_score_bureau_movel/') if f.endswith('.parquet')]
#df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]
#df_bureau = pd.concat(df_list, ignore_index=True)

df_bureau = spark.read.parquet(bucket_trusted)
df_bureau.show()

+--------------+----+---+---------------+------------------+----------------+--------+--------+----+-----------+------+
|       ts_proc| Ano|Mes|FLAG_INSTALACAO|ProductDescription|ProductMigration|SCORE_01|SCORE_02| FPD|    NUM_CPF| SAFRA|
+--------------+----+---+---------------+------------------+----------------+--------+--------+----+-----------+------+
|20260309213950|2025|  1|           true|               CMV|             PRE|     2.0|   557.0|   1|ZZZZZZZZY7Y|202501|
|20260309213950|2025|  1|           true|               CMV|             PRE|     2.0|   543.0|   0|ZZZZZZZZ787|202501|
|20260309213950|2025|  1|           true|               CMV|       Aquisição|     2.0|     1.0|   1|ZZZZZZZX7T9|202501|
|20260309213950|2025|  1|          false|               CMV|            NULL|   616.0|   654.0|NULL|ZZZZZZZNXYY|202501|
|20260309213950|2025|  1|          false|               CMV|            NULL|   596.0|   569.0|NULL|ZZZZZZZ9NTY|202501|
|20260309213950|2025|  1|           true

In [6]:
df_bureau.printSchema()

root
 |-- ts_proc: string (nullable = true)
 |-- Ano: integer (nullable = true)
 |-- Mes: integer (nullable = true)
 |-- FLAG_INSTALACAO: boolean (nullable = true)
 |-- ProductDescription: string (nullable = true)
 |-- ProductMigration: string (nullable = true)
 |-- SCORE_01: float (nullable = true)
 |-- SCORE_02: float (nullable = true)
 |-- FPD: integer (nullable = true)
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)



#### Ajustando os tipos de dados

In [ ]:
# Iremos ajustar os tipos de dados para otimizar a memoria e o correto processamento
df_bureau['Ano'] = df_bureau['Ano'].astype('int')
df_bureau['Mes'] = df_bureau['Mes'].astype('int')
df_bureau['FLAG_INSTALACAO'] = df_bureau['FLAG_INSTALACAO'].astype('bool')
df_bureau['ProductDescription'] = df_bureau['ProductDescription'].astype('object') #PROD
df_bureau['ProductMigration'] = df_bureau['ProductMigration'].astype('object') #flag_mig2
df_bureau['SCORE_01'] = df_bureau['SCORE_01'].astype('float32')
df_bureau['SCORE_02'] = df_bureau['SCORE_02'].astype('float32')
df_bureau['FPD'] = df_bureau['FPD'].astype('Int64')
df_bureau['NUM_CPF'] = df_bureau['NUM_CPF'].astype('object')
df_bureau['SAFRA'] = df_bureau['SAFRA'].astype('int')

#### Feature Engineer

Iremos criar apenas as variáveis:
- Relação entre `SCORE_01` e `SCORE_02`
- Média entre `SCORE_01` e `SCORE_02`
- Diferenca entre `SCORE_01` e `SCORE_02`
- Minino entre `SCORE_01` e `SCORE_02`

In [7]:
# Iremos criar as medidas descritas acima
df_bureau = df_bureau.select("*",
                             (col('SCORE_01') / col('SCORE_02')).alias('SCORE_RATE'),
                             ((col('SCORE_01') + col('SCORE_02')) / 2).alias('SCORE_AVG'),
                             (col('SCORE_02') - col('SCORE_01')).alias('SCORE_DIFF'),
                             least(col('SCORE_01'), col('SCORE_02')).alias('SCORE_MIN') #Menor valor
)
df_bureau.show(4)

+--------------+----+---+---------------+------------------+----------------+--------+--------+----+-----------+------+--------------------+---------+----------+---------+
|       ts_proc| Ano|Mes|FLAG_INSTALACAO|ProductDescription|ProductMigration|SCORE_01|SCORE_02| FPD|    NUM_CPF| SAFRA|          SCORE_RATE|SCORE_AVG|SCORE_DIFF|SCORE_MIN|
+--------------+----+---+---------------+------------------+----------------+--------+--------+----+-----------+------+--------------------+---------+----------+---------+
|20260309213950|2025|  1|           true|               CMV|             PRE|     2.0|   557.0|   1|ZZZZZZZZY7Y|202501|0.003590664272890485|    279.5|     555.0|      2.0|
|20260309213950|2025|  1|           true|               CMV|             PRE|     2.0|   543.0|   0|ZZZZZZZZ787|202501|0.003683241252302026|    272.5|     541.0|      2.0|
|20260309213950|2025|  1|           true|               CMV|       Aquisição|     2.0|     1.0|   1|ZZZZZZZX7T9|202501|                 2.0|

In [10]:
# Salvando o dataframe em parquet
df_bureau.write \
  .mode("overwrite") \
  .option("compression", "snappy") \
  .parquet(f"{bucket_feature_store}/book_variaveis_01")